In [5]:
import tkinter as tk
from tkinter import messagebox
from collections import Counter, defaultdict
import heapq

# Node class for Huffman Tree
class Node:
    def __init__(self, char, freq):
        self.char = char
        self.freq = freq
        self.left = None
        self.right = None

    def __lt__(self, other):
        return self.freq < other.freq

# Build Huffman Tree
def build_huffman_tree(text):
    frequency = Counter(text)
    priority_queue = [Node(char, freq) for char, freq in frequency.items()]
    heapq.heapify(priority_queue)

    while len(priority_queue) > 1:
        left = heapq.heappop(priority_queue)
        right = heapq.heappop(priority_queue)
        merged = Node(None, left.freq + right.freq)
        merged.left = left
        merged.right = right
        heapq.heappush(priority_queue, merged)

    return priority_queue[0] if priority_queue else None

# Generate Huffman codes
def generate_codes(root):
    codes = {}

    def generate_code(node, current_code):
        if node:
            if node.char is not None:
                codes[node.char] = current_code
            generate_code(node.left, current_code + "0")
            generate_code(node.right, current_code + "1")

    generate_code(root, "")
    return codes

# Encode text using Huffman codes
def huffman_encode(text, codes):
    return ''.join(codes[char] for char in text)

# Calculate compression ratio
def calculate_compression_ratio(original_text, encoded_text):
    original_size = len(original_text) * 8  # 1 character = 8 bits
    compressed_size = len(encoded_text)  # Compressed bit length
    return (1 - (compressed_size / original_size)) * 100

# GUI Application
class HuffmanCompressionApp:
    def __init__(self, root):
        self.root = root
        self.root.title("허프만 압축")
        
        # Entry for user text
        self.label = tk.Label(root, text="영어를 입력해주세요 :")
        self.label.pack(pady=5)
        
        self.text_entry = tk.Entry(root, width=50)
        self.text_entry.pack(pady=5)
        
        # Compress Button
        self.compress_button = tk.Button(root, text="압축 시작", command=self.compress_text)
        self.compress_button.pack(pady=10)

    def compress_text(self):
        text = self.text_entry.get().strip()
        if not text:
            messagebox.showwarning("입력 오류", "압축을 위해 글자를 입력해주세요.")
            return

        # Build Huffman Tree and get codes
        root = build_huffman_tree(text)
        huffman_codes = generate_codes(root)
        encoded_text = huffman_encode(text, huffman_codes)
        compression_ratio = calculate_compression_ratio(text, encoded_text)

        # Show results in a new window
        result_window = tk.Toplevel(self.root)
        result_window.title("허프만 압축 결과")

        # Display original and compressed information
        tk.Label(result_window, text="원문 :").pack(anchor='w', padx=10, pady=5)
        tk.Label(result_window, text=text).pack(anchor='w', padx=10, pady=5)
        
        tk.Label(result_window, text="허프만 코드 :").pack(anchor='w', padx=10, pady=5)
        for char, code in huffman_codes.items():
            tk.Label(result_window, text=f"'{char}': {code}").pack(anchor='w', padx=20)

        tk.Label(result_window, text="\n압축된 텍스트:").pack(anchor='w', padx=10, pady=5)
        tk.Label(result_window, text=encoded_text).pack(anchor='w', padx=10, pady=5)

        tk.Label(result_window, text=f"\nCompression Ratio: {compression_ratio:.2f}%").pack(anchor='w', padx=10, pady=10)

# Run the Tkinter application
root = tk.Tk()
app = HuffmanCompressionApp(root)
root.mainloop()
